In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

model_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

X = evaluation_data[model_features].copy()

print("Model input shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

display(X.head())

Model input shape: (74, 2)

Missing values:
steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
dtype: int64


,steps_deviation_7d_pct,sleep_deviation_7d_pct
7,-13.608883,55.643341
8,-23.071353,26.410835
9,11.806142,3.235515
10,14.010468,-9.668924
11,-4.515081,15.086531


In [2]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

model.fit(X)

evaluation_data["anomaly_label"] = model.predict(X)
evaluation_data["anomaly_score"] = model.decision_function(X)

print("Predicted anomalies:",
      (evaluation_data["anomaly_label"] == -1).sum())

print("\nPredicted normal observations:",
      (evaluation_data["anomaly_label"] == 1).sum())

display(
    evaluation_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "anomaly_label",
            "anomaly_score"
        ]
    ].head(10)
)

Predicted anomalies: 19

Predicted normal observations: 55


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,anomaly_label,anomaly_score
7,1503960366,2016-04-02,-13.608883,55.643341,-1,-0.046219
8,1503960366,2016-04-03,-23.071353,26.410835,1,0.058816
9,1503960366,2016-04-04,11.806142,3.235515,1,0.100964
10,1503960366,2016-04-05,14.010468,-9.668924,1,0.102085
11,1503960366,2016-04-09,-4.515081,15.086531,1,0.097566
12,1503960366,2016-04-10,-22.756449,32.468021,1,0.039307
13,1503960366,2016-04-11,-15.590472,-7.562077,1,0.113704
25,1927972279,2016-04-10,-31.608262,55.488592,-1,-0.037095
26,1927972279,2016-04-11,-60.719424,68.445975,-1,-0.124013
27,1927972279,2016-04-12,-99.220237,54.885923,-1,-0.151869


In [3]:

anomalies = evaluation_data[
    evaluation_data["anomaly_label"] == -1
].copy()

anomalies = anomalies.sort_values("anomaly_score")

display(
    anomalies[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "anomaly_score"
        ]
    ]
)

print("Unique participants flagged:",
      anomalies["Id"].nunique())

,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,anomaly_score
51,2347167796,2016-04-09,62.925707,-95.549339,-0.236835
115,4702921684,2016-04-09,67.628397,57.893258,-0.202775
176,7086361926,2016-04-10,-93.127545,-87.755102,-0.184810
27,1927972279,2016-04-12,-99.220237,54.885923,-0.151869
26,1927972279,2016-04-11,-60.719424,68.445975,-0.124013
78,4020332650,2016-04-10,-19.357157,-83.338694,-0.109558
49,2347167796,2016-04-07,4.557757,-84.689725,-0.105868
178,7086361926,2016-04-12,-94.806405,-3.854875,-0.069200
101,4445114986,2016-04-12,-94.208506,-20.239219,-0.065733
190,8378563200,2016-04-12,-68.580871,-49.413690,-0.062663


Unique participants flagged: 10


In [6]:


evaluation_data = evaluation_data.sort_values(
    ["Id", "day_number"]
).copy()

evaluation_data["steps_low"] = (
    evaluation_data["steps_deviation_7d_pct"] <= -10
)

evaluation_data["sleep_low"] = (
    evaluation_data["sleep_deviation_7d_pct"] <= -10
)

evaluation_data["steps_4_obs_alert"] = (
    evaluation_data.groupby("Id")["steps_low"]
    .transform(lambda s: s.rolling(4, min_periods=4).sum().eq(4))
)

evaluation_data["sleep_4_obs_alert"] = (
    evaluation_data.groupby("Id")["sleep_low"]
    .transform(lambda s: s.rolling(4, min_periods=4).sum().eq(4))
)

evaluation_data["rule_alert"] = (
    evaluation_data["steps_4_obs_alert"]
    | evaluation_data["sleep_4_obs_alert"]
)

print("Rule-based alerts:", evaluation_data["rule_alert"].sum())
print("ML anomalies:", (evaluation_data["anomaly_label"] == -1).sum())

Rule-based alerts: 3
ML anomalies: 19


In [10]:


ml_anomaly = evaluation_data["anomaly_label"] == -1
rule_alert = evaluation_data["rule_alert"]

print("ML anomalies:", ml_anomaly.sum())
print("Rule-based alerts:", rule_alert.sum())

print("Both methods flag:", (ml_anomaly & rule_alert).sum())
print("ML only:", (ml_anomaly & ~rule_alert).sum())
print("Rule only:", (~ml_anomaly & rule_alert).sum())

ML anomalies: 19
Rule-based alerts: 3
Both methods flag: 2
ML only: 17
Rule only: 1
